# PO-000 Interactive Falsification Sandbox

**Paper:** Projection Obstruction Anchor (PO-000)  
**AuthorityEffect:** `None`  
**Tolerance:** failures must exceed $10^{-12}$

This notebook is the Tier-2 **break-the-hypothesis** surface.  
Control cells should stay green. Break cells should **fail cleanly** when a false claim is asserted.

> **How to play.** Run all cells top-to-bottom. Then edit a break cell (change signs, aggregators, or $t$) and re-run — correct math stays within $10^{-12}$; wrong claims trip above it.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str((Path.cwd() if (Path.cwd() / "code").exists() else Path.cwd().parent).resolve() / "code"))

import numpy as np
from falsification_sandbox import (
    TOL,
    assert_all_breaks_trip,
    break_max_is_not_edge_additive,
    break_raw_equals_watched_without_observer,
    break_short_time_outside_window,
    break_unsigned_bell_formula,
    control_signed_bell_both_signs,
    run_all_breaks,
    short_time_lower_bound,
)

print(f"Imports OK. Publication tolerance TOL = {TOL}")

## 1. Control — signed Bell formula (both signs of $J$)

Closed form $D_{12} = 2|cs| + 2c^2s^2$ at $\theta = \pm\pi/8$ equals $1/\sqrt{2}+1/4$.

In [ ]:
pos, neg = control_signed_bell_both_signs()
expected = 1.0 / np.sqrt(2.0) + 0.25
print(f"+pi/8: {pos:.16f}")
print(f"-pi/8: {neg:.16f}")
print(f"expected: {expected:.16f}")
assert abs(pos - expected) <= TOL and abs(neg - expected) <= TOL
print("CONTROL PASSED: signed formula holds for both signs.")

## 2. BREAK — drop absolute value on $cs$

**False claim:** $D = 2cs + 2c^2s^2$ is valid for negative $\theta$.  
At $\theta=-\pi/8$ the unsigned expression is negative ($\approx -0.457$), so it cannot be a trace norm.

In [ ]:
r = break_unsigned_bell_formula(theta=-np.pi / 8)
print(r.detail)
print(f"residual = {r.residual:.6e}  (must be > {TOL})")
assert r.tripped, "Unsigned formula should disagree with the numerical replay"
print("BREAK TRIPPED AS EXPECTED: unsigned formula is not a valid closed form.")

## 3. BREAK — global max is not edge-additive

**False claim:** $\nu(\delta)=\max_e|\delta_e|$ is edge-additive (RE3 / P1 style).  
Counterexample: unit axes $e_1,e_2$ give $\max(e_1+e_2)=1 \neq 2 = \max e_1 + \max e_2$.

In [ ]:
r = break_max_is_not_edge_additive()
print(r.detail)
print(f"residual = {r.residual:.6e}")
assert r.tripped
print("BREAK TRIPPED AS EXPECTED: max aggregation fails edge additivity.")

## 4. BREAK — short-time lower bound is not global in $t$

**False claim:** $\nu t - 24g^2 t^2 \ge 0$ for all $t\ge 0$.  
At $(\nu,g,t)=(1,1,1)$ the expression equals $-23$.

In [ ]:
print("bound at t=0.01:", short_time_lower_bound(1.0, 1.0, 0.01))
print("bound at t=1.00:", short_time_lower_bound(1.0, 1.0, 1.0))
r = break_short_time_outside_window(nu_e=1.0, g=1.0, t=1.0)
print(r.detail)
assert r.tripped
print("BREAK TRIPPED AS EXPECTED: positivity is only a near-zero-window phenomenon.")

## 5. BREAK — raw $I$ is not watched $J$ without observers

**False claim:** $I(s)=\sum_\alpha J^\alpha(s)$ with no declared watch maps.  
Section 7.4.5: raw $I=6$ while watched losses can be $[0,0,0]$ (local-green / global-red).

In [ ]:
r = break_raw_equals_watched_without_observer(raw_I=6.0, watched_J=(0.0, 0.0, 0.0))
print(r.detail)
assert r.tripped
print("BREAK TRIPPED AS EXPECTED: raw tally and watched loss are distinct carriers.")

## 6. Full panel gate

In [ ]:
panel = run_all_breaks()
for item in panel:
    status = "TRIPPED" if item.tripped else "silent"
    print(f"[{status}] {item.name}: residual={item.residual:.3e} — {item.broken_hypothesis}")
assert_all_breaks_trip(panel)
print("\nALL EXPECTED BREAKS TRIPPED. Sandbox gate green.")

## 7. Your turn

Edit parameters below. Re-run and check whether residuals stay $\le 10^{-12}$ (control) or exceed it (break).

Ideas:
- Change `theta` in `break_unsigned_bell_formula`
- Change edge vectors in `break_max_is_not_edge_additive`
- Sweep $t$ in `break_short_time_outside_window` until the bound crosses zero

In [ ]:
# Interactive playground — edit freely
theta = -np.pi / 8
t_probe = 0.05
print(break_unsigned_bell_formula(theta=theta).detail)
print("short-time bound:", short_time_lower_bound(1.0, 1.0, t_probe))
print(break_short_time_outside_window(1.0, 1.0, t_probe))